<a href="https://colab.research.google.com/github/SahilPurabiya/PythonForAstronomy/blob/main/LombScargle%26ANOVA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Observation data : https://drive.google.com/file/d/1bg62TtI0WbvRUzlTQVnLS9t8hObIDcqh/view?usp=sharing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle

df = pd.read_csv("AAVSO.txt")   # (space before .txt)

# Clean column names (in case of extra spaces)
df.columns = df.columns.str.strip()

# Convert JD and Magnitude to numeric (force errors -> NaN)
df["JD"] = pd.to_numeric(df["JD"], errors="coerce")
df["Magnitude"] = pd.to_numeric(df["Magnitude"], errors="coerce")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle

# Read data
df = pd.read_csv("AAVSO.txt")

# V-band only
V = df[df["Band"] == "V"].copy()

# Select one night (your range)
night = V[(V["JD"] >= 2461053.15) & (V["JD"] <= 2461053.27)].copy()

# Arrays
t = night["JD"].to_numpy()
y = night["Magnitude"].to_numpy()

# Remove NaNs
mask = np.isfinite(t) & np.isfinite(y)
t = t[mask]
y = y[mask]

# Remove mean
y = y - np.mean(y)

# Period range (days) ~ 70 to 100 minutes
P_MIN = 0.01
P_MAX = 0.09

# Frequency range (cycles/day)
f_min = 1.0 / P_MAX
f_max = 1.0 / P_MIN

# High resolution grid
N_FREQ = 5000
frequency = np.linspace(f_min, f_max, N_FREQ)

# Lomb–Scargle power
ls = LombScargle(t, y)
power = ls.power(frequency)

# Best period
best_idx = np.argmax(power)
best_freq = frequency[best_idx]
best_period = 1.0 / best_freq

print("Best Period (days):", best_period)
print("Best Period (minutes):", best_period * 24 * 60)

# Plot periodogram
plt.figure(figsize=(8, 5))
plt.plot(1.0 / frequency, power, color="black")
plt.axvline(best_period, color="red", linestyle="--", linewidth=2)
plt.xlabel("Period (days)")
plt.ylabel("LS Power")
plt.title("Lomb–Scargle Periodogram – Single Night (HV Vir)")
plt.tight_layout()
plt.show()


In [ ]:
# V-band only (CV)
V = df[df["Band"].astype(str).str.strip() == "V"].copy()

# Select your time range
night = V[(V["JD"] >= 2461053.15) & (V["JD"] <= 2461053.27)].copy()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------
# MULTI-HARMONIC AoV (Peranso style)
# ---------------------------

def aov_multiharmonic(t, y, freq, nharm=4):
    N = len(t)
    if N < (2 * nharm + 2):
        return 0.0

    w = 2.0 * np.pi * freq

    # Design matrix: [1, cos(wt), sin(wt), cos(2wt), sin(2wt), ...]
    X = np.ones((N, 2 * nharm + 1))
    for k in range(1, nharm + 1):
        X[:, 2 * k - 1] = np.cos(k * w * t)
        X[:, 2 * k]     = np.sin(k * w * t)

    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    y_fit = X @ beta

    y_mean = np.mean(y)
    ss_tot = np.sum((y - y_mean) ** 2)
    ss_res = np.sum((y - y_fit) ** 2)
    ss_mod = ss_tot - ss_res

    if ss_res <= 1e-30 or ss_tot <= 1e-30:
        return 0.0

    p = X.shape[1]
    df1 = p - 1
    df2 = N - p
    if df2 <= 0:
        return 0.0

    F = (ss_mod / df1) / (ss_res / df2)
    return F


def aov_periodogram_peak(t, y, P_min, P_max, nfreq=5000, nharm=4, plot=True):
    fmin = 1.0 / P_max
    fmax = 1.0 / P_min
    freqs = np.linspace(fmin, fmax, nfreq)

    power = np.zeros_like(freqs)
    for i, f in enumerate(freqs):
        power[i] = aov_multiharmonic(t, y, f, nharm=nharm)

    periods = 1.0 / freqs

    # -------- Peak detection (BEST one) --------
    best_idx = np.argmax(power)
    best_period = periods[best_idx]
    best_power = power[best_idx]

    # Sort periods for correct axis direction
    sort_idx = np.argsort(periods)
    periods_sorted = periods[sort_idx]
    power_sorted = power[sort_idx]

    if plot:
        plt.figure(figsize=(10, 5))
        plt.plot(periods_sorted, power_sorted, lw=1)
        plt.axvline(best_period, linestyle="--", linewidth=1.5)

        plt.xlabel("Period (days)")
        plt.ylabel("AoV Power (F-statistic)")
        plt.title(f"ANOVA (AoV) Periodogram  | Peak Period = {best_period:.6f} d")
        plt.grid(True, alpha=0.3)
        plt.show()

    return best_period, best_power


# ---------------------------
# LOAD YOUR AAVSO FILE
# ---------------------------

filename = r"AAVSO.txt"
df = pd.read_csv(filename)

df = df.rename(columns={"Magnitude": "Mag", "Uncertainty": "Err"})

data = df[df["Band"].isin(["V", "CV"])].copy()
data = data.dropna(subset=["JD", "Mag"])

# ---------------------------
# FILTER JD RANGE (optional)
# ---------------------------

JD_min = 2461053.15
JD_max = 2461053.27
data = data[(data["JD"] >= JD_min) & (data["JD"] <= JD_max)].copy()

print("Total points used:", len(data))

t = data["JD"].to_numpy()
y = data["Mag"].to_numpy()

# Remove mean
y = y - np.mean(y)


# ---------------------------
# RUN PEAK SEARCH
# ---------------------------

P_min = 0.01
P_max = 0.09
nfreq = 5000
nharm = 4   # Peranso: "Nbr harmonics = 4"

peak_period, peak_power = aov_periodogram_peak(
    t, y,
    P_min=P_min,
    P_max=P_max,
    nfreq=nfreq,
    nharm=nharm,
    plot=True
)

print("\n===== PEAK RESULT =====")
print(f"Best Peak Period = {peak_period:.8f} days")
print(f"Best Period in minutes = {peak_period * 1440:.6f} minutes")

